In [15]:
# ============================================================
# 🚀 OFFLINE BERT Multilingual NER Training Notebook (WORKS IN YOUR ENV)
# ------------------------------------------------------------
# - Uses local model folder: ./mbert/
# - Uses FAST tokenizer (tokenizer.json ~1.9 MB)
# - Compatible with Transformers 2.x TrainingArguments (eval_strategy)
# - Supports .word_ids() for correct label alignment
# ============================================================

%pip install transformers datasets seqeval accelerate -q

import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    BertTokenizerFast,
    BertForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
)
from seqeval.metrics import accuracy_score, f1_score

# ------------------------------------------------------------
# 1. LOAD GDPR TRAINING DATA (CSV)
# ------------------------------------------------------------

df = pd.read_csv("ner_training_data_bio.csv")

sentences = df.groupby("sentence_id")["token"].apply(list).tolist()
labels_raw = df.groupby("sentence_id")["label"].apply(list).tolist()

# ------------------------------------------------------------
# 2. BUILD LABEL → ID MAPPING
# ------------------------------------------------------------

label_list = sorted(df["label"].unique())
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

labels_encoded = [[label2id[l] for l in seq] for seq in labels_raw]

dataset = Dataset.from_dict({
    "tokens": sentences,
    "ner_tags": labels_encoded,
})

dataset = dataset.train_test_split(test_size=0.1)

# ------------------------------------------------------------
# 3. LOAD LOCAL MBERT TOKENIZER & MODEL (FAST TOKENIZER!)
# ------------------------------------------------------------

LOCAL_MODEL_PATH = "./mbert"

tokenizer = BertTokenizerFast.from_pretrained(
    LOCAL_MODEL_PATH,
    local_files_only=True
)

model = BertForTokenClassification.from_pretrained(
    LOCAL_MODEL_PATH,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    local_files_only=True
)

# ------------------------------------------------------------
# 4. TOKENIZATION + LABEL ALIGNMENT
# ------------------------------------------------------------

def tokenize_and_align_labels(batch):
    tokenized = tokenizer(
        batch["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    aligned_labels = []
    for i, label_seq in enumerate(batch["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        prev = None
        aligned = []

        for w in word_ids:
            if w is None:
                aligned.append(-100)
            elif w != prev:
                aligned.append(label_seq[w])
            else:
                aligned.append(label_seq[w])
            prev = w

        aligned_labels.append(aligned)

    tokenized["labels"] = aligned_labels
    return tokenized

tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

data_collator = DataCollatorForTokenClassification(tokenizer)

# ------------------------------------------------------------
# 5. OFFLINE METRICS
# ------------------------------------------------------------

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    true_preds = []
    true_labels = []

    for pred_seq, label_seq in zip(preds, labels):
        p_seq, l_seq = [], []
        for p, l in zip(pred_seq, label_seq):
            if l != -100:
                p_seq.append(id2label[p])
                l_seq.append(id2label[l])
        true_preds.append(p_seq)
        true_labels.append(l_seq)

    return {
        "accuracy": accuracy_score(true_labels, true_preds),
        "f1": f1_score(true_labels, true_preds),
    }

# ------------------------------------------------------------
# 6. TRAINING CONFIG (COMPATIBLE WITH TRANSFORMERS 2.x)
# ------------------------------------------------------------

training_args = TrainingArguments(
    output_dir="./ml-ner-model",
    learning_rate=4e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=4,
    weight_decay=0.01,

    # IMPORTANT: Your environment uses OLD signature → eval_strategy, NOT evaluation_strategy
    eval_strategy="epoch"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# ------------------------------------------------------------
# 7. TRAIN MODEL
# ------------------------------------------------------------

trainer.train()

# ------------------------------------------------------------
# 8. SAVE TRAINED MODEL
# ------------------------------------------------------------

model.save_pretrained("ml-ner-model")
tokenizer.save_pretrained("ml-ner-model")

print("🎉 Training complete! Model saved to ml-ner-model/")


Note: you may need to restart the kernel to use updated packages.


The tokenizer you are loading from './mbert' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Some weights of BertForTokenClassification were not initialized from the model checkpoint at ./mbert and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 500/500 [00:00<00:00, 39509.27 examples/s]
C:\Users\andreea.asimine\AppData\Local\Temp\ipykernel_23968\532915173.py:146: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
c:\Users\andreea.asimine\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\d

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.047100,0.000155,1.000000,1.000000
2,0.000300,0.000087,1.000000,1.000000
3,0.000200,0.000066,1.000000,1.000000
4,0.000100,0.000060,1.000000,1.000000


c:\Users\andreea.asimine\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\andreea.asimine\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\andreea.asimine\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\andreea.asimine\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then de

🎉 Training complete! Model saved to ml-ner-model/
